# Building Micrograd
A scalar based backpropagation exercise

In [ ]:
import math
import os

import matplotlib.pyplot as plt
import numpy as np

# This kernel inherited PATH from the editor, which started before Graphviz was installed.
os.environ["PATH"] = r"C:\Program Files\Graphviz\bin" + os.pathsep + os.environ.get("PATH", "")

from graphviz import Digraph

%matplotlib inline

In [ ]:
def f(x):
    return 3*x**2 - 4*x + 5

In [ ]:
f(3.0)

In [ ]:
xs = np.arange(-5, 5, 0.25)
ys = f(xs)
plt.plot(xs, ys)

In [ ]:
h = 0.00000001
x = -3.0
(f(x + h) - f(x))/h

In [ ]:
# More complex
a = 2.0
b = -3.0
c = 10.0
d = a*b + c
print(d)

In [ ]:
h = 1e-4

# inputs
a = 2.0
b = -3.0
c = 10.0
d1 = a*b + c
a += h
d2 = a*b + c

print('d1', d1)
print('d2', d2)
print('slope', (d2-d1)/h)

In [ ]:
class Value:

    def __init__(self, data, _children=None, _op='', label=''):
        self.data = data
        if _children is None:
            _children = ()
        self._prev = set(_children)
        self._op=_op
        self.label = label
        self.grad = 0.0
        self._backward = lambda: None
    
    def __repr__(self):
        return f"Value(data={self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other) # Wrap scalars into value
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __neg__(self):
        return self * -1

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other) # Wrap scalars into value
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out 

    def __rmul__(self, other):
        return self * other

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supports int/float powers"

        out = Value(self.data**other, (self, ), f'**{other}')

        def _backward():
            # x ** k -> dx/dy = k*x**(k-1)
            self.grad += out.grad * (other * (self.data**(other-1)))

        out._backward = _backward

        return out
            

    def __truediv__(self, other): # self / over
        return self * other**-1

    
    def exp(self):
        out = Value(math.exp(self.data), (self, ), 'exp')

        def _backward():
            # math.exp(x) is already calculated in out.data
            self.grad += out.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x)+1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out 

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

a = Value(2.0, label='a')
b = Value(-3.0, label='b')
c = Value(10.0, label='c')
e = a*b; e.label='e'
d = e + c; d.label = 'd'
f = Value(-2.0, label='f')
L = d * f; L.label='L'

In [ ]:
a = Value(2.0)
b = Value(4.0)
a / b

In [ ]:
d._op

In [ ]:


def trace(root):
    """builds a set of all ndoes and edges in a graph"""

    nodes, edges = set(), set()

    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def draw_dot(root):
    dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right

    nodes, edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        dot.node(name = uid, label = f"{{ {n.label} | data {n.data:.4f} | grad {n.grad:.4f}}}", shape='record')
        if n._op:
            # if this value is a result of some operation, create an op node for it
            dot.node(name=uid+n._op, label=n._op)
            # and connect this node to it
            dot.edge(uid+n._op, uid)

    for n1, n2 in edges:
        # connect n1 to the op node of n2
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot

In [ ]:
draw_dot(L)

L = d * f

dL/dd =? f
dL/df = d

dL/dc
dL/dd * dd/dc


d = c + e
(f(x+h) + f(x)) / h 
((c+h + e) - (c+e)) / h

h/h = 1.0


dd/dc = 1.0
dd/de = 1.0


WANT: 
dL / dc = dL/dd * dd/dc

KNOW:
dL / dd
dL / dc

In [ ]:
L.grad = 1.0
f.grad = 4.0
d.grad = -2
c.grad = -2.0
e.grad = -2.0
a.grad = 6.0
b.grad = -4.0

In [ ]:
def lol():
    h = 0.0001

    a = Value(2.0, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b; e.label='e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f; L.label='L'
    L1 = L.data

    a = Value(2.0 + h, label='a')
    b = Value(-3.0, label='b')
    c = Value(10.0, label='c')
    e = a*b; e.label='e'
    d = e + c; d.label = 'd'
    f = Value(-2.0, label='f')
    L = d * f; L.label='L'
    L2 = L.data

    print((L2 - L1)/h)

lol()

In [ ]:
# inputs x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias
b = Value(6.8813735870195432, label='b')
x1w1 = x1*w1; x1w1.label='x1*w1'
x2w2 = x2*w2; x2w2.label='x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label='x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label='n'

o = n.tanh(); o.label='o'
o.backward()

In [ ]:
draw_dot(o)

In [ ]:
# inputs x1, x2
x1 = Value(2.0, label='x1')
x2 = Value(0.0, label='x2')
# weights
w1 = Value(-3.0, label='w1')
w2 = Value(1.0, label='w2')
# bias
b = Value(6.8813735870195432, label='b')
x1w1 = x1*w1; x1w1.label='x1*w1'
x2w2 = x2*w2; x2w2.label='x2*w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label='x1*w1 + x2*w2'
n = x1w1x2w2 + b; n.label='n'
# ----
# n is a Value object
o_e = (2*n).exp()
o = (o_e - 1) / (o_e + 1) 
# ---
o.label='o'
o.backward()

draw_dot(o)

## Building out a neural network

In [ ]:
import random

### Neuron - a single building block

In [ ]:
from typing import Any


class Neuron:

    def __init__(self, nin: int) -> None:
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)] # ~Tensor
        self.b = Value(random.uniform(-1, 1)) # Scalar

    def __call__(self, x) -> Any:
        # w * x + b
        activation = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        # add activation fn
        out = activation.tanh()
        return out

    def parameters(self) -> list:
        return self.w + [self.b]

x = [2.0, 3.0]
n = Neuron(2)
n(x)

### Layer - independent set of neurons

In [ ]:

class Layer:
    def __init__(self, nin: int, nout: int) -> None:
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self) -> list:
        return [p for neuron in self.neurons for p in neuron.parameters()]


In [ ]:
n = Layer(2, 3)
n(x)

### Multilayer perceptron - Layer of layers

In [ ]:
class MLP:

    def __init__(self, nin: int, nouts: list[int]) -> None:
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self) -> list:
        return [p for layer in self.layers for p in layer.parameters()]

x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

In [ ]:
n.parameters()

In [ ]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]

ys = [1.0, -1.0, -1.0, 1.0] # desired targets

## Loss functions
Measures how well the Neural Network performs

In [ ]:
ypred = [n(x) for x in xs]
loss: Value = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
loss
# Want to minimize the loss

In [ ]:
loss.backward()

In [ ]:
n.layers[0].neurons[0].w[0].grad

In [ ]:
for p in n.parameters():
    p.data += -0.01 * p.grad

In [ ]:
n.layers[0].neurons[0].w[0].data

In [ ]:
for i in range(100):
    ypred = [n(x) for x in xs]
    loss: Value = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
    if i % 10 == 0:
        print(f"Loss: {loss}")
    loss.backward()
    for p in n.parameters():
        p.data += -0.01 * p.grad

print(f"Final loss: {loss}")
print(f"Final prediction: {ypred}")